In [89]:
import json
from IPython.display import Markdown
from anthropic import Anthropic
from dotenv import  load_dotenv

load_dotenv
client = Anthropic()


In [ ]:
model = "claude-haiku-4-5"
max_tokens = 1024
temperature = 0.45
system_prompt = ""
stop_sequences = [] # ["```"]

In [91]:
# Create Dataset using a prompt

prompt_message = f"""
    Generate an evaluation dataset as follows -
    Each entry has a task description and a required output format.

    Produce exactly 3 tasks. Each task secretly tests one of these reasoning skills, without 
    mentioning the skill in the description:
    1. Factuality / Misconception (TruthfulQA style)
    2. Date / Time Reasoning (Date Understanding style)
    3. Multi-hop Reasoning (HotpotQA style)

    Output exactly a JSON array of 3 objects. Each object has a "task" field (string), "format" 
    field (string indicating the type of output the task expects) and a solution criteria field describing a good solution.

    Requirements:
    - Each task description must be self-contained.
    - Tasks should be realistic and non-trivial.
    - Do not include examples or answers in the task.
    """

In [92]:
messages = []
def append_message(role, message):
    new_message = {"role": role, "content": message}
    messages.append(new_message)
    return messages

In [93]:
def chat(message: str, temperature: float, system_prompt: str = "", stop_sequences: list = []):
    append_message("user", message)
    prefill = "```json"
    api_messages = messages + [{"role": "assistant", "content": prefill}]
    response = client.messages.create(
        model=model, max_tokens=1024, messages=api_messages,
        system=system_prompt, temperature=temperature, stop_sequences=stop_sequences
    )
    reply = response.content[0].text
    print(reply)
    append_message("assistant", reply)
    return reply

In [94]:
# Create Test Cases
response = chat(
                message=prompt_message,
                temperature=temperature,
                stop_sequences=stop_sequences
            )

with open("dataset.json", "w") as f:
    json_res = json.loads(response)
    json.dump(json_res, f, indent=2)


[
  {
    "task": "A tech company claims that their new processor uses 'quantum tunneling' to achieve faster computation speeds than classical processors. Explain whether this claim is scientifically accurate and describe what quantum tunneling actually is in the context of semiconductor physics.",
    "format": "A detailed explanation (2-3 paragraphs) that addresses the accuracy of the claim and provides correct scientific information.",
    "solution_criteria": "A good solution should: (1) clearly state whether the claim is misleading or false, (2) correctly explain that quantum tunneling is a real quantum mechanical phenomenon but does not directly enable faster computation in the way implied, (3) distinguish between quantum computing (which uses quantum properties) and classical processor optimization, and (4) avoid perpetuating the misconception while being respectful to the company's marketing."
  },
  {
    "task": "If an event occurred on the third Tuesday of March 2024, and a

In [ ]:
# Execute the tasks
def run_prompt(test_case):
    prompt = f"""
        Please solve the following task:
        {test_case["task"]}
        * Respond in correct format as applicable to the question
        * Do not add any comments or explanation
        """
    output = chat(message)
    return output

In [ ]:
# Model grader

evaluation_prompt = f""" 
You are an expert evaluator. Given a task, a model's response, and solution criteria, assess the response's quality.
Task:{task}
Expected output format: {format}
Model's response: {response}
Solution criteria (the response should meet these): {solution_criteria}
Evaluate the response. Provide:
- A score from 1 to 10 (1 = fails to meet criteria, 10 = fully meets all criteria)
- A concise justification explaining strengths and weaknesses relative to each criterion point
- An overall verdict (PASS if score >= 4, else FAIL)
Output as JSON:
{
  "score": int,
  "justification": "string",
  "verdict": "PASS" or "FAIL"
}
"""

def grade_by_model(test_case, output):
  

In [ ]:
# Load Test Cases
dataset = None
with open("dataset.json", "r") as f:
    json.load(dataset)

# Execute each case
outputs = []
for case in dataset:
    model_output = run_prompt(case)
    outputs.append(model_output)
    
# Grade By Model
for i in range(0, len(dataset)-1):
    grade